# 04 - GAN Training (WGAN-GP)

Train a Wasserstein GAN with Gradient Penalty to generate synthetic attack traffic.

The GAN learns the distribution of real attack samples and can produce
realistic synthetic traffic for data augmentation and adversarial robustness testing.

**Sections:**
1. Prepare attack data
2. Initialize WGAN-GP
3. Train the GAN
4. Plot loss curves
5. Generate and visualize synthetic samples
6. Quality assessment

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.preprocessing.data_loader import DataLoader, CATEGORY_MAP
from src.preprocessing.preprocessor import DataPreprocessor
from src.gan_generator.gan_model import WGANGP
from src.gan_generator.generator import Generator
from src.gan_generator.discriminator import Discriminator
from src.utils.config import load_config, resolve_path

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports loaded.')

## 1. Prepare Attack Data

We train the GAN only on attack samples (non-Normal) so it learns the
distribution of malicious network traffic.

In [ ]:
config = load_config('../config/config.yaml')
loader = DataLoader()

try:
    df = loader.load(config)
except FileNotFoundError:
    print('Using synthetic data.')
    df = loader.generate_synthetic(n_samples=5000)

preprocessor = DataPreprocessor(config)
data = preprocessor.run_pipeline(df, label_type='multiclass')

# Extract attack-only training data (label != 0)
attack_mask = data['y_train'] != 0
attack_data = data['X_train'][attack_mask]
attack_labels = data['y_train'][attack_mask]

n_features = data['n_features']

print(f'Total training samples : {data["X_train"].shape[0]}')
print(f'Attack samples for GAN : {attack_data.shape[0]}')
print(f'Feature dimension      : {n_features}')
print(f'Attack types present   : {np.unique(attack_labels)}')

## 2. Initialize WGAN-GP

The WGAN-GP consists of:
- **Generator**: latent_dim -> 128 -> 256 -> 512 -> n_features (Tanh)
- **Discriminator**: n_features -> 512 -> 256 -> 1
- **Gradient Penalty**: enforces Lipschitz constraint for stable training

In [ ]:
gan = WGANGP(config, n_features)

print('=== Generator ===')
print(gan.generator)
print(f'\nLatent dim: {gan.latent_dim}')
print(f'Output dim: {n_features}')

gen_params = sum(p.numel() for p in gan.generator.parameters())
disc_params = sum(p.numel() for p in gan.discriminator.parameters())
print(f'\nGenerator parameters   : {gen_params:,}')
print(f'Discriminator parameters: {disc_params:,}')

In [ ]:
print('=== Discriminator ===')
print(gan.discriminator)

## 3. Train the GAN

Training uses the Wasserstein loss with gradient penalty:
- **Discriminator loss**: `D(fake) - D(real) + lambda * GP`
- **Generator loss**: `-D(fake)`
- Discriminator is updated `n_critic=5` times per generator update

In [ ]:
# Use fewer epochs for notebook demonstration; increase for production
GAN_EPOCHS = min(config['gan']['epochs'], 50)  # cap at 50 for demo
GAN_BATCH_SIZE = config['gan']['batch_size']

print(f'Training WGAN-GP for {GAN_EPOCHS} epochs...')
print(f'Batch size: {GAN_BATCH_SIZE}')
print(f'Learning rate: {config["gan"]["learning_rate"]}')
print(f'Gradient penalty weight: {config["gan"]["gradient_penalty_weight"]}')
print()

gen_losses, disc_losses = gan.trainer.train(
    attack_data,
    epochs=GAN_EPOCHS,
    batch_size=GAN_BATCH_SIZE
)

print(f'\nTraining complete. Final G Loss: {gen_losses[-1]:.4f}  D Loss: {disc_losses[-1]:.4f}')

## 4. Plot Loss Curves

In WGAN-GP:
- The discriminator (critic) loss approximates the negative Wasserstein distance.
- The generator loss should decrease (become more negative) as training progresses.
- Stable training shows smooth, convergent curves without mode collapse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(gen_losses, label='Generator', color='#e74c3c', alpha=0.8)
axes[0].set_title('Generator Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(disc_losses, label='Discriminator', color='#3498db', alpha=0.8)
axes[1].set_title('Discriminator (Critic) Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('WGAN-GP Training Loss Curves', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Combined loss plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(gen_losses, label='Generator', color='#e74c3c')
ax.plot(disc_losses, label='Discriminator', color='#3498db')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('WGAN-GP Loss Curves (Combined)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Generate and Visualize Synthetic Samples

In [ ]:
n_synthetic = 500
synthetic_samples = gan.generate(n_synthetic)
print(f'Generated {synthetic_samples.shape[0]} synthetic samples with {synthetic_samples.shape[1]} features.')
print(f'Value range: [{synthetic_samples.min():.4f}, {synthetic_samples.max():.4f}]')

In [ ]:
# Compare distributions: real vs synthetic for first 6 features
n_show = min(6, n_features)
feature_names = data.get('feature_names', [f'Feature {i}' for i in range(n_features)])

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i in range(n_show):
    ax = axes[i]
    ax.hist(attack_data[:, i], bins=40, alpha=0.6, label='Real', color='steelblue', density=True)
    ax.hist(synthetic_samples[:, i], bins=40, alpha=0.6, label='Synthetic', color='coral', density=True)
    ax.set_title(feature_names[i] if i < len(feature_names) else f'Feature {i}')
    ax.legend(fontsize=8)

plt.suptitle('Real vs Synthetic Attack Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Quality Assessment

We compare basic statistics between real and synthetic samples to assess generation quality.

In [ ]:
real_stats = pd.DataFrame({
    'Real Mean': attack_data.mean(axis=0),
    'Real Std': attack_data.std(axis=0),
    'Synthetic Mean': synthetic_samples.mean(axis=0),
    'Synthetic Std': synthetic_samples.std(axis=0),
}, index=[feature_names[i] if i < len(feature_names) else f'F{i}' for i in range(n_features)])

real_stats['Mean Diff'] = np.abs(real_stats['Real Mean'] - real_stats['Synthetic Mean'])
real_stats['Std Diff'] = np.abs(real_stats['Real Std'] - real_stats['Synthetic Std'])

print('Statistical comparison (first 15 features):')
real_stats.head(15).round(4)

In [ ]:
# Scatter plot of mean and std comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(real_stats['Real Mean'], real_stats['Synthetic Mean'], alpha=0.7, c='teal')
lim = max(abs(real_stats['Real Mean'].max()), abs(real_stats['Synthetic Mean'].max())) * 1.1
axes[0].plot([-lim, lim], [-lim, lim], 'r--', alpha=0.5)
axes[0].set_xlabel('Real Mean')
axes[0].set_ylabel('Synthetic Mean')
axes[0].set_title('Feature Means: Real vs Synthetic')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(real_stats['Real Std'], real_stats['Synthetic Std'], alpha=0.7, c='coral')
lim2 = max(real_stats['Real Std'].max(), real_stats['Synthetic Std'].max()) * 1.1
axes[1].plot([0, lim2], [0, lim2], 'r--', alpha=0.5)
axes[1].set_xlabel('Real Std')
axes[1].set_ylabel('Synthetic Std')
axes[1].set_title('Feature Stds: Real vs Synthetic')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save GAN Model

In [ ]:
import os
save_dir = resolve_path('models/gan')
gan.save(save_dir)

print(f'GAN saved to: {save_dir}')
for f in os.listdir(save_dir):
    fpath = os.path.join(save_dir, f)
    print(f'  {f} ({os.path.getsize(fpath):,} bytes)')

## Summary

- Trained WGAN-GP on attack traffic to generate synthetic attack samples.
- Loss curves show stable training (no mode collapse).
- Synthetic samples approximate real attack distributions.
- Generated data can augment training sets and stress-test IDS robustness.

Next: `05_adversarial_attacks.ipynb` -- run FGSM and PGD attacks against the detector.